[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/02_Introduction_to_ONNX/01_What_is_ONNX/What_is_ONNX_Apply.ipynb)

# 1.1 What is ONNX? — Apply

## Table of Contents
1. [Environment Setup and Verification](#section-1)
2. [Building Your First ONNX Model from Scratch](#section-2)
3. [Inspecting Model Structure Programmatically](#section-3)
4. [Running Inference with ONNX Runtime](#section-4)
5. [Visualizing Computation Graphs](#section-5)
6. [Exporting a PyTorch Model to ONNX](#section-6)
7. [Comparing Framework and ONNX Outputs](#section-7)
8. [Model Serialization and Deserialization](#section-8)
9. [Performance Benchmark: Framework vs ONNX Runtime](#section-9)
10. [Exercises](#section-10)

In [ ]:
# Install required packages
!pip install onnx onnxruntime numpy matplotlib networkx torch -q

<a id='section-1'></a>
## Section 1: Environment Setup and Verification

Before building ONNX models, we need to verify our environment. The key packages in the ONNX ecosystem are:

- **`onnx`**: The core library for creating, loading, and validating ONNX models
- **`onnxruntime`**: Microsoft's high-performance inference engine
- **`numpy`**: Numerical arrays that interface with ONNX tensors
- **`torch`**: PyTorch for demonstrating model export

Each ONNX model targets a specific **IR version** and **opset version**. The IR version defines the structure of the protobuf, while the opset version defines which operators are available and their semantics.

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np
import torch
import sys

print("Environment Verification")
print("=" * 60)
print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"ONNX:         {onnx.__version__}")
print(f"ONNX Runtime: {ort.__version__}")
print(f"PyTorch:      {torch.__version__}")
print(f"\nONNX IR Version: {onnx.IR_VERSION}")
print(f"ONNX Max OpSet:  {onnx.defs.onnx_opset_version()}")
print(f"\nORT Available Providers: {ort.get_available_providers()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device:  {torch.cuda.get_device_name(0)}")

<a id='section-2'></a>
## Section 2: Building Your First ONNX Model from Scratch

We will build an ONNX model entirely from the Python API — no PyTorch or TensorFlow needed. This demonstrates that ONNX is a **standalone format**, not tied to any framework.

Our target function:

$$f(x) = \sigma(W_2 \cdot \text{ReLU}(W_1 \cdot x + b_1) + b_2)$$

where $\sigma$ is the sigmoid function, implementing a 2-layer neural network with:
- Input: $x \in \mathbb{R}^4$
- Hidden: $h \in \mathbb{R}^8$  
- Output: $y \in \mathbb{R}^1$ (binary classification probability)

### Building Blocks

The ONNX Python API provides these core constructors:
- `helper.make_tensor_value_info(name, type, shape)` — defines a tensor's metadata
- `helper.make_node(op_type, inputs, outputs, **attrs)` — creates an operator node
- `helper.make_graph(nodes, name, inputs, outputs, initializer)` — assembles the DAG
- `helper.make_model(graph, opset_imports)` — wraps the graph in a model container

In [ ]:
import numpy as np
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model

# Network dimensions
INPUT_DIM = 4
HIDDEN_DIM = 8
OUTPUT_DIM = 1

# Initialize weights with Xavier initialization
np.random.seed(42)
W1_data = (np.random.randn(INPUT_DIM, HIDDEN_DIM) * np.sqrt(2.0 / INPUT_DIM)).astype(np.float32)
b1_data = np.zeros(HIDDEN_DIM, dtype=np.float32)
W2_data = (np.random.randn(HIDDEN_DIM, OUTPUT_DIM) * np.sqrt(2.0 / HIDDEN_DIM)).astype(np.float32)
b2_data = np.zeros(OUTPUT_DIM, dtype=np.float32)

# Create ONNX tensor initializers
W1_init = numpy_helper.from_array(W1_data, name='W1')
b1_init = numpy_helper.from_array(b1_data, name='b1')
W2_init = numpy_helper.from_array(W2_data, name='W2')
b2_init = numpy_helper.from_array(b2_data, name='b2')

# Define computation graph nodes
# Layer 1: h = ReLU(x @ W1 + b1)
node_matmul1 = helper.make_node('MatMul', ['X', 'W1'], ['pre_h1'])
node_add1 = helper.make_node('Add', ['pre_h1', 'b1'], ['h1_linear'])
node_relu = helper.make_node('Relu', ['h1_linear'], ['h1'])

# Layer 2: y = Sigmoid(h @ W2 + b2)
node_matmul2 = helper.make_node('MatMul', ['h1', 'W2'], ['pre_h2'])
node_add2 = helper.make_node('Add', ['pre_h2', 'b2'], ['h2_linear'])
node_sigmoid = helper.make_node('Sigmoid', ['h2_linear'], ['Y'])

# Define input and output tensors
X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, ['batch', INPUT_DIM])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, ['batch', OUTPUT_DIM])

# Assemble the graph
graph = helper.make_graph(
    nodes=[node_matmul1, node_add1, node_relu, node_matmul2, node_add2, node_sigmoid],
    name='binary_classifier',
    inputs=[X_info],
    outputs=[Y_info],
    initializer=[W1_init, b1_init, W2_init, b2_init]
)

# Create the model
model = helper.make_model(
    graph,
    opset_imports=[helper.make_opsetid('', 17)]
)
model.ir_version = 8
model.producer_name = 'onnx-tutorial'
model.doc_string = 'A 2-layer binary classifier built from scratch'

# Validate
check_model(model)
print("✓ Model created and validated successfully!")
print(f"\nModel Summary:")
print(f"  Nodes:        {len(model.graph.node)}")
print(f"  Parameters:   {sum(np.prod(numpy_helper.to_array(i).shape) for i in model.graph.initializer)}")
print(f"  Input shape:  ['batch', {INPUT_DIM}]")
print(f"  Output shape: ['batch', {OUTPUT_DIM}]")

# Save to disk
onnx.save(model, 'binary_classifier.onnx')
print(f"\n  Saved to: binary_classifier.onnx")
import os
print(f"  File size: {os.path.getsize('binary_classifier.onnx')} bytes")

<a id='section-3'></a>
## Section 3: Inspecting Model Structure Programmatically

ONNX models are fully introspectable. Every component — nodes, edges, tensors, shapes, types — can be queried programmatically. This is crucial for debugging, optimization, and understanding what an exported model actually contains.

The inspection hierarchy follows the protobuf structure:

```
model.ir_version          → IR version number
model.graph.name          → Graph name
model.graph.input         → List of ValueInfoProto (graph inputs)
model.graph.output        → List of ValueInfoProto (graph outputs)
model.graph.initializer   → List of TensorProto (weights)
model.graph.node          → List of NodeProto (operators)
```

In [ ]:
# Load and inspect the model
import onnx
from onnx import numpy_helper, shape_inference

model = onnx.load('binary_classifier.onnx')

print("═" * 70)
print("MODEL INSPECTION REPORT")
print("═" * 70)

# Top-level metadata
print(f"\n┌─ Metadata ─────────────────────────────────────────────────────────┐")
print(f"│  IR Version:     {model.ir_version:<50}│")
print(f"│  Producer:       {model.producer_name:<50}│")
print(f"│  OpSet Version:  {model.opset_import[0].version:<50}│")
print(f"│  Doc String:     {model.doc_string:<50}│")
print(f"└────────────────────────────────────────────────────────────────────┘")

# Graph inputs
print(f"\n┌─ Graph Inputs ─────────────────────────────────────────────────────┐")
for inp in model.graph.input:
    tensor_type = inp.type.tensor_type
    elem_type = TensorProto.DataType.Name(tensor_type.elem_type)
    shape = [d.dim_param or d.dim_value for d in tensor_type.shape.dim]
    print(f"│  {inp.name:<12} type={elem_type:<10} shape={str(shape):<25}│")
print(f"└────────────────────────────────────────────────────────────────────┘")

# Graph outputs
print(f"\n┌─ Graph Outputs ────────────────────────────────────────────────────┐")
for out in model.graph.output:
    tensor_type = out.type.tensor_type
    elem_type = TensorProto.DataType.Name(tensor_type.elem_type)
    shape = [d.dim_param or d.dim_value for d in tensor_type.shape.dim]
    print(f"│  {out.name:<12} type={elem_type:<10} shape={str(shape):<25}│")
print(f"└────────────────────────────────────────────────────────────────────┘")

# Initializers (weights)
print(f"\n┌─ Initializers (Parameters) ────────────────────────────────────────┐")
total_params = 0
for init in model.graph.initializer:
    arr = numpy_helper.to_array(init)
    total_params += arr.size
    print(f"│  {init.name:<8} shape={str(list(arr.shape)):<15} "
          f"dtype={arr.dtype}  params={arr.size:<8}│")
print(f"│  {'TOTAL':<8} {'':15} {'':18} params={total_params:<8}│")
print(f"└────────────────────────────────────────────────────────────────────┘")

# Nodes
print(f"\n┌─ Computation Nodes ────────────────────────────────────────────────┐")
for i, node in enumerate(model.graph.node):
    inputs_str = ', '.join(node.input)
    outputs_str = ', '.join(node.output)
    print(f"│  [{i}] {node.op_type:<10} inputs=[{inputs_str:<20}] → [{outputs_str}]")
print(f"└────────────────────────────────────────────────────────────────────┘")

<a id='section-4'></a>
## Section 4: Running Inference with ONNX Runtime

ONNX Runtime (ORT) is the reference high-performance inference engine. It implements all ONNX operators with optimized kernels for CPU, GPU (CUDA, ROCm), and specialized accelerators (TensorRT, OpenVINO, DirectML).

The inference pipeline:

```
┌──────────────────┐     ┌───────────────────────┐     ┌──────────────────┐
│  Load .onnx file │────▶│  Create InferenceSession │──▶│  session.run()   │
│  (model bytes)   │     │  (graph optimization)    │   │  (execute graph) │
└──────────────────┘     └───────────────────────┘     └──────────────────┘
```

Key concepts:
- **Session**: A compiled, optimized representation of the graph
- **Execution Provider (EP)**: Backend that runs the operators (CPU, CUDA, etc.)
- **IO Binding**: Advanced API for zero-copy GPU inference

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Create inference session
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession('binary_classifier.onnx', session_options,
                                providers=['CPUExecutionProvider'])

# Inspect session
print("Inference Session Info:")
print(f"  Provider: {session.get_providers()}")
print(f"  Inputs:")
for inp in session.get_inputs():
    print(f"    {inp.name}: shape={inp.shape}, type={inp.type}")
print(f"  Outputs:")
for out in session.get_outputs():
    print(f"    {out.name}: shape={out.shape}, type={out.type}")

# Run inference with single sample
x_single = np.random.randn(1, 4).astype(np.float32)
result = session.run(None, {'X': x_single})
print(f"\nSingle sample inference:")
print(f"  Input:  {x_single.round(4)}")
print(f"  Output: {result[0].round(6)} (probability)")
print(f"  Class:  {'Positive' if result[0][0, 0] > 0.5 else 'Negative'}")

# Run batch inference
batch_size = 100
x_batch = np.random.randn(batch_size, 4).astype(np.float32)
result_batch = session.run(None, {'X': x_batch})
print(f"\nBatch inference (n={batch_size}):")
print(f"  Output shape: {result_batch[0].shape}")
print(f"  Mean prob:    {result_batch[0].mean():.4f}")
print(f"  Positive:     {(result_batch[0] > 0.5).sum()} / {batch_size}")

# Timing benchmark
n_runs = 1000
start = time.perf_counter()
for _ in range(n_runs):
    session.run(None, {'X': x_single})
elapsed = time.perf_counter() - start
print(f"\nPerformance ({n_runs} runs):")
print(f"  Total time:    {elapsed*1000:.1f} ms")
print(f"  Per inference: {elapsed/n_runs*1000:.3f} ms")
print(f"  Throughput:    {n_runs/elapsed:.0f} inferences/sec")

<a id='section-5'></a>
## Section 5: Visualizing Computation Graphs

Understanding the structure of an ONNX model is greatly aided by visualization. We can render the computation graph as a node-edge diagram showing the flow of data through operators.

The visualization maps ONNX concepts to visual elements:
- **Operator nodes** → Blue rectangles (with op_type label)
- **Input/Output tensors** → Green/Orange ellipses  
- **Initializers (weights)** → Red diamonds
- **Data flow edges** → Directed arrows with tensor names

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import onnx
from onnx import numpy_helper

def visualize_onnx_graph(model, figsize=(14, 10)):
    """Visualize an ONNX model as a computation graph."""
    G = nx.DiGraph()
    graph = model.graph
    
    # Track tensor producers
    init_names = {i.name for i in graph.initializer}
    input_names = {i.name for i in graph.input if i.name not in init_names}
    output_names = {o.name for o in graph.output}
    
    # Add operator nodes
    node_labels = {}
    for i, node in enumerate(graph.node):
        node_id = f"op_{i}"
        node_labels[node_id] = f"{node.op_type}\n#{i}"
        G.add_node(node_id, ntype='operator')
        
        # Connect inputs
        for inp in node.input:
            if inp in init_names:
                G.add_node(inp, ntype='initializer')
                node_labels[inp] = inp
            elif inp in input_names:
                G.add_node(inp, ntype='input')
                node_labels[inp] = inp
            G.add_edge(inp, node_id)
        
        # Connect outputs
        for out in node.output:
            if out in output_names:
                G.add_node(out, ntype='output')
                node_labels[out] = out
                G.add_edge(node_id, out)
            else:
                G.add_node(out, ntype='intermediate')
                node_labels[out] = out
                G.add_edge(node_id, out)
    
    # Layout
    pos = nx.spring_layout(G, seed=42, k=2)
    
    # Try topological layout
    try:
        layers = list(nx.topological_generations(G))
        pos = {}
        for layer_idx, layer in enumerate(layers):
            for node_idx, node in enumerate(sorted(layer)):
                pos[node] = (node_idx - len(layer)/2, -layer_idx)
    except:
        pass
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Draw by type
    color_map = {'operator': '#6699CC', 'initializer': '#FF9999', 
                 'input': '#99FF99', 'output': '#FFCC66', 'intermediate': '#DDDDDD'}
    
    for ntype, color in color_map.items():
        nodes = [n for n in G.nodes() if G.nodes[n].get('ntype') == ntype]
        if nodes:
            nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=color,
                                   node_size=1500 if ntype == 'operator' else 1000,
                                   node_shape='s' if ntype == 'operator' else 'o',
                                   ax=ax)
    
    nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=15,
                           edge_color='gray', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, node_labels, ax=ax, font_size=7)
    
    import matplotlib.patches as mpatches
    legend = [mpatches.Patch(color=c, label=t.capitalize()) for t, c in color_map.items()]
    ax.legend(handles=legend, loc='upper right')
    ax.set_title(f"ONNX Graph: {graph.name}", fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

model = onnx.load('binary_classifier.onnx')
visualize_onnx_graph(model)

<a id='section-6'></a>
## Section 6: Exporting a PyTorch Model to ONNX

The most common way to create ONNX models in practice is by **exporting** from a training framework. PyTorch provides `torch.onnx.export()` which traces the model's computation graph and translates it to ONNX operators.

The export process works by:
1. Running a forward pass with dummy inputs to trace the computation
2. Recording all operations in PyTorch's internal IR (TorchScript or FX graph)
3. Translating each PyTorch op to equivalent ONNX operator(s)
4. Serializing the graph + weights to a `.onnx` file

In [ ]:
import torch
import torch.nn as nn

# Define a more complex PyTorch model
class MultiLayerClassifier(nn.Module):
    def __init__(self, input_dim=10, hidden_dims=[64, 32], output_dim=5):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Create and export model
torch_model = MultiLayerClassifier()
torch_model.eval()

# Dummy input for tracing
dummy_input = torch.randn(1, 10)

# Export to ONNX
torch.onnx.export(
    torch_model,
    dummy_input,
    'pytorch_classifier.onnx',
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'logits': {0: 'batch_size'}
    },
    opset_version=17,
    do_constant_folding=True
)

# Verify exported model
exported_model = onnx.load('pytorch_classifier.onnx')
onnx.checker.check_model(exported_model)

print("PyTorch → ONNX Export Successful!")
print(f"\nPyTorch Model:")
total_params = sum(p.numel() for p in torch_model.parameters())
print(f"  Parameters: {total_params:,}")
print(f"  Architecture: {[10, 64, 32, 5]}")

print(f"\nONNX Model:")
print(f"  Nodes: {len(exported_model.graph.node)}")
print(f"  Initializers: {len(exported_model.graph.initializer)}")
print(f"  OpSet: {exported_model.opset_import[0].version}")

print(f"\nOperator breakdown:")
from collections import Counter
op_counts = Counter(n.op_type for n in exported_model.graph.node)
for op, count in op_counts.most_common():
    print(f"  {op:<25} × {count}")

<a id='section-7'></a>
## Section 7: Comparing Framework and ONNX Outputs

A critical validation step after export is verifying **numerical equivalence** between the original framework model and the ONNX model. Due to floating-point arithmetic, we expect outputs to match within a small tolerance:

$$\|y_{\text{PyTorch}} - y_{\text{ONNX}}\|_\infty < \epsilon \quad \text{where } \epsilon \approx 10^{-6}$$

We use both:
- **Absolute tolerance**: $|a - b| < \text{atol}$
- **Relative tolerance**: $|a - b| < \text{rtol} \cdot |b|$

In [ ]:
import onnxruntime as ort
import numpy as np
import matplotlib.pyplot as plt

# Run same inputs through both PyTorch and ONNX Runtime
n_samples = 500
test_inputs = np.random.randn(n_samples, 10).astype(np.float32)

# PyTorch inference
torch_model.eval()
with torch.no_grad():
    pytorch_outputs = torch_model(torch.from_numpy(test_inputs)).numpy()

# ONNX Runtime inference
session = ort.InferenceSession('pytorch_classifier.onnx',
                                providers=['CPUExecutionProvider'])
ort_outputs = session.run(None, {'input': test_inputs})[0]

# Compare
abs_diff = np.abs(pytorch_outputs - ort_outputs)
rel_diff = abs_diff / (np.abs(pytorch_outputs) + 1e-10)

print("Numerical Equivalence Check")
print("=" * 50)
print(f"Max absolute difference: {abs_diff.max():.2e}")
print(f"Mean absolute difference: {abs_diff.mean():.2e}")
print(f"Max relative difference: {rel_diff.max():.2e}")
print(f"Mean relative difference: {rel_diff.mean():.2e}")

matches = np.allclose(pytorch_outputs, ort_outputs, atol=1e-5, rtol=1e-5)
print(f"\nAll close (atol=1e-5, rtol=1e-5): {'✓ PASS' if matches else '✗ FAIL'}")

# Visualize the differences
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter plot of outputs
axes[0].scatter(pytorch_outputs.flatten(), ort_outputs.flatten(), alpha=0.3, s=10)
lims = [min(pytorch_outputs.min(), ort_outputs.min()),
        max(pytorch_outputs.max(), ort_outputs.max())]
axes[0].plot(lims, lims, 'r--', lw=1, label='y = x (perfect match)')
axes[0].set_xlabel('PyTorch Output')
axes[0].set_ylabel('ONNX Runtime Output')
axes[0].set_title('Output Correlation')
axes[0].legend()

# Histogram of absolute differences
axes[1].hist(abs_diff.flatten(), bins=50, color='steelblue', edgecolor='navy', alpha=0.7)
axes[1].axvline(abs_diff.mean(), color='red', linestyle='--', label=f'Mean: {abs_diff.mean():.1e}')
axes[1].set_xlabel('Absolute Difference')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of |PyTorch - ORT|')
axes[1].legend()
axes[1].set_yscale('log')

# Per-output-dim comparison
per_dim_diff = abs_diff.mean(axis=0)
axes[2].bar(range(len(per_dim_diff)), per_dim_diff, color='coral', edgecolor='darkred')
axes[2].set_xlabel('Output Dimension')
axes[2].set_ylabel('Mean Absolute Diff')
axes[2].set_title('Error by Output Dimension')

plt.suptitle('PyTorch vs ONNX Runtime: Numerical Equivalence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Model Serialization and Deserialization

ONNX models are serialized using **Protocol Buffers** (protobuf), Google's binary serialization format. This provides:

- **Compact binary encoding**: Much smaller than JSON/XML
- **Schema evolution**: Forward/backward compatible versioning
- **Language-neutral**: Parseable from C++, Python, Java, etc.
- **Memory mapping**: Large models can be memory-mapped for efficient loading

For very large models (>2GB protobuf limit), ONNX supports **external data** format where weights are stored in separate files.

In [ ]:
import onnx
import os
import time

# Different serialization formats
model = onnx.load('pytorch_classifier.onnx')

# 1. Standard binary protobuf (.onnx)
onnx.save(model, 'model_binary.onnx')
binary_size = os.path.getsize('model_binary.onnx')

# 2. Text protobuf format (human-readable, for debugging)
with open('model_text.pbtxt', 'w') as f:
    f.write(str(model))
text_size = os.path.getsize('model_text.pbtxt')

# 3. Serialized bytes (for in-memory transfer)
model_bytes = model.SerializeToString()
bytes_size = len(model_bytes)

# Benchmark load times
n_loads = 100

start = time.perf_counter()
for _ in range(n_loads):
    _ = onnx.load('model_binary.onnx')
binary_load_time = (time.perf_counter() - start) / n_loads

start = time.perf_counter()
for _ in range(n_loads):
    m = onnx.ModelProto()
    m.ParseFromString(model_bytes)
load_from_bytes_time = (time.perf_counter() - start) / n_loads

print("Serialization Comparison")
print("=" * 60)
print(f"{'Format':<25} {'Size':<15} {'Load Time':<15}")
print("─" * 60)
print(f"{'Binary protobuf (.onnx)':<25} {binary_size:>8} bytes  {binary_load_time*1000:>8.3f} ms")
print(f"{'Text protobuf (.pbtxt)':<25} {text_size:>8} bytes  {'N/A':>8}")
print(f"{'In-memory bytes':<25} {bytes_size:>8} bytes  {load_from_bytes_time*1000:>8.3f} ms")
print(f"\nText/Binary size ratio: {text_size/binary_size:.1f}×")
print(f"  (Text is {text_size/binary_size:.1f}× larger — binary is more efficient for storage)")

# Clean up
os.remove('model_text.pbtxt')

<a id='section-9'></a>
## Section 9: Performance Benchmark — Framework vs ONNX Runtime

One of the key motivations for ONNX is **inference performance**. ONNX Runtime applies graph optimizations (operator fusion, constant folding, memory planning) that often yield significant speedups over raw framework execution.

We benchmark:
- **PyTorch (eager mode)**: Default execution, no compilation
- **ONNX Runtime (CPU)**: With all graph optimizations enabled
- **Various batch sizes**: To observe throughput scaling

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

batch_sizes = [1, 4, 16, 64, 256, 1024]
n_warmup = 50
n_benchmark = 200

pytorch_times = []
ort_times = []

session = ort.InferenceSession('pytorch_classifier.onnx',
                                providers=['CPUExecutionProvider'])

for bs in batch_sizes:
    x_np = np.random.randn(bs, 10).astype(np.float32)
    x_torch = torch.from_numpy(x_np)
    
    # Warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = torch_model(x_torch)
        _ = session.run(None, {'input': x_np})
    
    # Benchmark PyTorch
    start = time.perf_counter()
    for _ in range(n_benchmark):
        with torch.no_grad():
            _ = torch_model(x_torch)
    pt_time = (time.perf_counter() - start) / n_benchmark * 1000
    pytorch_times.append(pt_time)
    
    # Benchmark ORT
    start = time.perf_counter()
    for _ in range(n_benchmark):
        _ = session.run(None, {'input': x_np})
    ort_time = (time.perf_counter() - start) / n_benchmark * 1000
    ort_times.append(ort_time)

# Results
print("Performance Benchmark: PyTorch vs ONNX Runtime")
print("=" * 65)
print(f"{'Batch Size':<12} {'PyTorch (ms)':<15} {'ORT (ms)':<15} {'Speedup':<10}")
print("─" * 65)
for i, bs in enumerate(batch_sizes):
    speedup = pytorch_times[i] / ort_times[i]
    print(f"{bs:<12} {pytorch_times[i]:<15.3f} {ort_times[i]:<15.3f} {speedup:<10.2f}×")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(batch_sizes))
width = 0.35

ax1.bar(x_pos - width/2, pytorch_times, width, label='PyTorch', color='#FF6B6B', edgecolor='darkred')
ax1.bar(x_pos + width/2, ort_times, width, label='ONNX Runtime', color='#4ECDC4', edgecolor='darkgreen')
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Inference Latency Comparison')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(batch_sizes)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

speedups = [pt/ort for pt, ort in zip(pytorch_times, ort_times)]
ax2.plot(batch_sizes, speedups, 'o-', color='purple', linewidth=2, markersize=8)
ax2.axhline(y=1.0, color='gray', linestyle='--', label='Break-even')
ax2.fill_between(batch_sizes, 1, speedups, alpha=0.2, color='purple')
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Speedup (PyTorch time / ORT time)')
ax2.set_title('ONNX Runtime Speedup Factor')
ax2.set_xscale('log', base=2)
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('PyTorch vs ONNX Runtime Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-10'></a>
## Section 10: Exercises

### Exercise 1: Build a Custom ONNX Model
Create an ONNX model that computes the **L2 norm** of a vector:
$$\|x\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}$$

Hint: Use `Mul`, `ReduceSum`, and `Sqrt` operators.

### Exercise 2: Dynamic Batch Support
Modify the binary classifier model to support **dynamic batch sizes** using symbolic dimensions.

### Exercise 3: Multi-Output Model
Build an ONNX model with two outputs — both the class probabilities and the logits (pre-softmax values).

### Exercise 4: Performance Profiling
Use `onnxruntime.InferenceSession` with profiling enabled (`session_options.enable_profiling = True`) to identify bottleneck operators.

In [ ]:
# Exercise 1 Solution: L2 Norm ONNX Model
from onnx import helper, TensorProto, numpy_helper
import numpy as np

# ||x||_2 = sqrt(sum(x * x))
X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [None])  # dynamic length
Y_info = helper.make_tensor_value_info('norm', TensorProto.FLOAT, [1])

# Nodes: X → Mul(X,X) → ReduceSum → Sqrt → norm
mul_node = helper.make_node('Mul', ['X', 'X'], ['X_squared'])
reduce_node = helper.make_node('ReduceSum', ['X_squared'], ['sum_sq'], keepdims=1)
sqrt_node = helper.make_node('Sqrt', ['sum_sq'], ['norm'])

graph = helper.make_graph(
    [mul_node, reduce_node, sqrt_node],
    'l2_norm', [X_info], [Y_info]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
onnx.checker.check_model(model)

# Test
import onnxruntime as ort
onnx.save(model, 'l2_norm.onnx')
sess = ort.InferenceSession('l2_norm.onnx', providers=['CPUExecutionProvider'])

x_test = np.array([3.0, 4.0], dtype=np.float32)
result = sess.run(None, {'X': x_test})[0]
expected = np.linalg.norm(x_test)

print(f"Exercise 1: L2 Norm Model")
print(f"  Input: {x_test}")
print(f"  ONNX output:    {result[0]:.6f}")
print(f"  NumPy expected: {expected:.6f}")
print(f"  Match: {'✓' if np.isclose(result[0], expected) else '✗'}")

# Clean up temp files
import os
for f in ['binary_classifier.onnx', 'pytorch_classifier.onnx', 'model_binary.onnx', 'l2_norm.onnx']:
    if os.path.exists(f):
        os.remove(f)

---

**Next:** [Why ONNX Matters — Deep Dive](../02_Why_ONNX_Matters/Why_ONNX_Matters_Deep_Dive.ipynb)